In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [2]:
!python -V

Python 3.10.13


In [3]:
import pickle
import pandas as pd

In [5]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [6]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [7]:
year = 2023
month = 3

df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet')

In [8]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

# Question 1

What's the standard deviation of the predicted duration for this dataset?

Answer: 6.2474897941993985

In [9]:
std_pred = pd.Series(y_pred).std()

print(std_pred)

6.2474897941993985


# Q2. Preparing the output

What's the size of the output file?

Answer: 65.46 MB

In [10]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'prediction': y_pred
})



In [13]:
import os

output_file = f'output/yellow/{year:04d}-{month:02d}.parquet'

# Create the output directory if it doesn't exist
os.makedirs('output/yellow', exist_ok=True)

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [22]:
# Get the file size
file_size = os.path.getsize(output_file)
file_size = file_size / 1024 / 1024
print(f"File size: {round(file_size, 2)} MB") 

File size: 65.46 MB


# Q3. Creating the scoring script

Answer: jupyter nbconvert --to script starter.ipynb

# Q4. Virtual environment

What's the first hash for the Scikit-Learn dependency?
Answer: sha256:057b991ac64b3e75c9c04b5f9395eaf19a6179244c089afdebaad98264bff37c

# Q5. Parametrize the script

Run the script for April 2023.
What's the mean predicted duration?
Answer: 14.292282936862449

# Q6. Docker container

Now run the script with docker. What's the mean predicted duration for May 2023?
Answer: 0.19174419265916945